# Exemplo 01 - Cibersegurança: detectando links maliciosos

### Palestra Machine Learning e IA: Da máquina que aprende à IA que age

### Prof. Dr. Ahirton Lopes (https://github.com/ahirtonlopes)

**Onde isso aparece no mercado:** Filtros de e-mail, navegadores e gateways de segurança corporativos classificam milhões de links por dia para bloquear phishing antes que alguém clique. Um modelo como este é a primeira barreira: ele olha só para o texto da URL e decide se ela parece maliciosa.

**Conceito da palestra:** Aprendizado supervisionado: o modelo aprende a partir de exemplos que já vêm com a resposta certa (URL maliciosa ou benigna).

### Conceitos Fundamentais

Neste exemplo, estamos abordando o **Aprendizado Supervisionado**.

**Aprendizado Supervisionado** é um tipo de machine learning onde o algoritmo aprende a partir de dados rotulados, ou seja, dados que já possuem a 'resposta' correta associada. O objetivo é que o modelo aprenda a mapear uma entrada para uma saída, de forma que, quando novos dados sem rótulo forem apresentados, ele consiga prever a saída corretamente.

No nosso caso:
*   **Entrada (features)**: São as URLs.
*   **Saída (rótulo/label)**: É se a URL é 'bad' (maliciosa) ou 'good' (benigna).

O modelo 'aprende' a relação entre as características das URLs e seus rótulos, e então usa esse conhecimento para classificar novas URLs.

### Como aproveitar este exemplo

Este é o **primeiro exemplo da palestra**, e ele vem de propósito antes de vários conceitos que você vai aprofundar ao longo da graduação. A ideia é simples: ver um modelo funcionando de ponta a ponta em poucos minutos, do dado cru até a previsão, para depois destrinchar cada etapa com calma.

Você vai esbarrar aqui em três coisas que merecem um estudo mais detalhado depois:

*   **divisão treino/teste (holdout)**, que tem versões mais robustas como validação cruzada, K-pastas e Leave One Out;
*   **acurácia**, que anda junto com precisão, recall, F1 e matriz de confusão (veja o Exemplo 02);
*   **regressão logística**, que é um dos modelos clássicos de Machine Learning.

Não se preocupe em dominar tudo agora. Cada célula traz uma explicação curta do que está acontecendo, e o objetivo do momento é pegar a intuição do fluxo completo. Os detalhes vêm depois, e aí este exemplo vai fazer muito mais sentido em retrospecto.

In [1]:
# Import de nossas bibliotecas

import pandas as pd
import numpy as np
import random
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

In [2]:
# Baixando a base de dados direto do repositório da palestra para o Google Colab
# Fonte original: https://www.kaggle.com/antonyj453/urldataset

!curl -O https://raw.githubusercontent.com/ahirtonlopes/machine-learning-e-ia-2026/main/Datasets/url_data.csv

In [3]:
# ALTERNATIVA: se preferir subir o arquivo na mão, descomente as linhas abaixo,
# execute esta célula e selecione o url_data.csv do seu computador.

# from google.colab import files
#
# uploaded = files.upload()
#
# for fn in uploaded.keys():
#     print('Arquivo "{name}" com {length} bytes'.format(
#         name=fn, length=len(uploaded[fn])))

In [4]:
# Lendo dados a partir de nosso arquivo .csv

data = pd.read_csv("url_data.csv")
data.head()

,url,label
0,diaryofagameaddict.com,bad
1,espdesign.com.au,bad
2,iamagameaddict.com,bad
3,kalantzis.net,bad
4,slightlyoffcenter.net,bad


### Entendendo Nosso Conjunto de Dados (Dataset)

O arquivo `url_data.csv` contém duas colunas principais:

*   **`url`**: A URL em si, que é a nossa *feature* (característica) de entrada para o modelo.
*   **`label`**: O rótulo associado à URL, indicando se ela é 'bad' (maliciosa) ou 'good' (benigna). Este é o nosso *target* (alvo) que o modelo tentará prever.

In [5]:
# Rótulos
y = data["label"]

# Atributos (por enquanto, somente URL)
url_list = data["url"]

### O que é TF-IDF?

**TF-IDF** (Term Frequency-Inverse Document Frequency) é uma técnica estatística utilizada para avaliar a importância de uma palavra em relação a um documento em uma coleção de documentos (corpus).

*   **Term Frequency (TF)**: Mede a frequência com que um termo aparece em um documento. Quanto mais uma palavra aparece em um documento, mais importante ela tende a ser *para aquele documento*.
*   **Inverse Document Frequency (IDF)**: Mede a raridade de um termo em todo o conjunto de documentos. Palavras que aparecem em muitos documentos (como 'a', 'o', 'de') têm um IDF baixo, pois não são muito informativas. Palavras raras que aparecem em poucos documentos têm um IDF alto, indicando que são mais específicas e informativas.

Combinando TF e IDF, o TF-IDF dá um peso maior a palavras que são frequentes em um documento, mas raras na coleção geral de documentos. No nosso caso, o `TfidfVectorizer` transforma cada URL em um vetor numérico, onde cada dimensão do vetor representa a importância de uma palavra (ou sequência de caracteres) naquela URL.

In [6]:
# Tokenização (via TfidfVectorizer do Scikit-learn - Ref. https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html)
vectorizer = TfidfVectorizer()

In [7]:
# Guardando vetores gerados em uma variável X que funcionará como nossos Xatributos
X = vectorizer.fit_transform(url_list)

In [8]:
# Visualizando as dimensões dos nossos dados após a vetorização
print(f"Shape de X (features vetorizadas): {X.shape}")
print(f"Shape de y (rótulos): {y.shape}")

# Onde X.shape[0] é o número de URLs e X.shape[1] é o número total de termos (palavras/caracteres) únicos encontrados pelo TF-IDF.

Shape de X (features vetorizadas): (420464, 423927)
Shape de y (rótulos): (420464,)


In [9]:
# Divisão em sets de treinamento/teste (Regra de Pareto - 80/20 - Ref. https://pt.wikipedia.org/wiki/Princ%C3%ADpio_de_Pareto)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### Divisão de Dados em Treinamento e Teste

É uma prática essencial em Machine Learning dividir nossos dados em dois conjuntos:

1.  **Conjunto de Treinamento (Training Set)**: Usado para 'ensinar' o modelo. O algoritmo aprenderá padrões e relações a partir desses dados.
2.  **Conjunto de Teste (Test Set)**: Usado para *avaliar* o desempenho do modelo em dados que ele *nunca viu* antes. Isso nos ajuda a entender quão bem o modelo generaliza para novas informações e evita o *overfitting* (onde o modelo memoriza os dados de treinamento, mas não se sai bem com dados novos).

A regra de Pareto (80/20) é uma heurística comum, onde 80% dos dados são usados para treinamento e 20% para teste, mas essa proporção pode variar dependendo do tamanho e características do dataset.

> Estamos usando aqui a forma mais simples de separar os dados, chamada **holdout**: um único sorteio, uma parte para treinar e outra para testar. Existem formas mais robustas (validação cruzada, K-pastas, Leave One Out) e por que elas costumam dar uma estimativa mais confiável do desempenho. Por enquanto, o holdout basta para ver o modelo funcionando.

In [10]:
# Visualizando as dimensões dos conjuntos de treinamento e teste
print(f"Shape de X_train (features de treinamento): {X_train.shape}")
print(f"Shape de X_test (features de teste): {X_test.shape}")
print(f"Shape de y_train (rótulos de treinamento): {y_train.shape}")
print(f"Shape de y_test (rótulos de teste): {y_test.shape}")

Shape de X_train (features de treinamento): (336371, 423927)
Shape de X_test (features de teste): (84093, 423927)
Shape de y_train (rótulos de treinamento): (336371,)
Shape de y_test (rótulos de teste): (84093,)


In [11]:
# Construção de nosso modelo usando Regressão Logística (Ref. https://edisciplinas.usp.br/pluginfile.php/3769787/mod_resource/content/1/09_RegressaoLogistica.pdf)
logit = LogisticRegression()
logit.fit(X_train, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following

### Regressão Logística para Classificação

**Regressão Logística** é um algoritmo de classificação, apesar do nome conter 'regressão'. Ele é muito utilizado para prever a probabilidade de uma ocorrência (por exemplo, se uma URL é maliciosa ou não) com base em um conjunto de características de entrada.

Neste caso, ele aprende a relação entre os vetores TF-IDF das URLs (X_train) e seus rótulos (y_train) para determinar a probabilidade de uma URL ser 'bad' ou 'good'. Se a probabilidade for maior que um certo limiar (geralmente 0.5), ele classifica como 'bad', caso contrário como 'good'.

In [12]:
# Acurácia total de nosso modelo
print("A acurácia do nosso modelo é: ",logit.score(X_test, y_test))

A acurácia do nosso modelo é:  0.9640635962565255


### O que significa a Acurácia?

A **Acurácia** é uma métrica de avaliação de modelos que mede a proporção de previsões corretas (tanto positivas quanto negativas) em relação ao número total de previsões feitas. Em termos simples, ela nos diz a porcentagem de vezes que o nosso modelo acertou a classificação.

Uma acurácia de 0.964, como a que obtemos, significa que o modelo classificou corretamente aproximadamente 96.4% das URLs no conjunto de teste.

> A acurácia é a métrica mais intuitiva e por isso é a primeira que aprendemos, mas ela está longe de ser suficiente. A próxima célula já mostra o primeiro motivo, e o Exemplo 02 (detecção de fraude) acrescenta precisão, recall e a matriz de confusão.

### 96% é bom? Compare sempre com um baseline

Antes de comemorar, vale a pergunta que sempre deve vir junto com qualquer métrica: **bom comparado com o quê?**

A acurácia tem uma armadilha conhecida: quando as classes estão desbalanceadas, um modelo que não aprendeu nada pode parecer excelente só por chutar sempre a classe mais frequente. Se 99% das URLs de uma base fossem benignas, um "modelo" que respondesse **good** para tudo teria 99% de acurácia e seria completamente inútil, porque nunca detectaria uma URL maliciosa.

Por isso a referência mínima (o *baseline*) é sempre o resultado do chute mais burro possível. Vamos ver como está a distribuição da nossa base e quanto esse chute renderia:

In [13]:
from sklearn.dummy import DummyClassifier

# Como as classes estão distribuídas na nossa base?
print("Distribuição das classes:")
print(data['label'].value_counts())
print()
print("Em proporção:")
print(data['label'].value_counts(normalize=True).round(4))
print()

# O baseline: responder sempre a classe mais frequente, ignorando a URL
baseline = DummyClassifier(strategy='most_frequent').fit(X_train, y_train)

acc_modelo   = logit.score(X_test, y_test)
acc_baseline = baseline.score(X_test, y_test)

print(f"Acurácia do nosso modelo : {acc_modelo:.4f}")
print(f"Acurácia do baseline     : {acc_baseline:.4f}  (chutando sempre 'good')")
print(f"Ganho sobre o baseline   : {acc_modelo - acc_baseline:.4f}")

Distribuição das classes:
label
good    344821
bad      75643
Name: count, dtype: int64

Em proporção:
label
good    0.8201
bad     0.1799
Name: proportion, dtype: float64

Acurácia do nosso modelo : 0.9641
Acurácia do baseline     : 0.8221  (chutando sempre 'good')
Ganho sobre o baseline   : 0.1420


Agora o número faz sentido de verdade.

A base está desbalanceada: cerca de **82% das URLs são benignas**. Ou seja, um chute fixo em `good` já entregaria 82% de acurácia sem olhar uma única URL. Nosso modelo chega a 96,4%, então o ganho real sobre o baseline é de aproximadamente **14 pontos percentuais**, e não os 96 que a leitura ingênua sugeriria.

A lição que fica: **acurácia sozinha não basta em base desbalanceada**. É por isso que existem precisão, recall, F1 e a matriz de confusão, que separam o desempenho por classe e mostram quantas URLs maliciosas o modelo realmente pegou. Essas métricas aparecem no Exemplo 02 (detecção de fraude).

### Testando o Modelo com Novas URLs

Agora que nosso modelo foi treinado e avaliado, podemos usá-lo para prever o rótulo de novas URLs que ele nunca viu.

In [14]:
# URLs de exemplo para testar
new_urls = [
    'malicious-site.phishing.ru', # Esperado: bad
    'example.com/safe_page',    # Esperado: good
    'free-robux-generator.xyz', # Esperado: bad
    'github.com/my-project'     # Esperado: good
]

# Primeiro, precisamos vetorizar as novas URLs usando o MESMO vetorizador que treinamos
new_urls_vectorized = vectorizer.transform(new_urls)

# Agora, fazemos as previsões
predictions = logit.predict(new_urls_vectorized)

# Exibindo os resultados
print("--- Previsões para Novas URLs ---")
for url, pred in zip(new_urls, predictions):
    print(f"URL: {url} -> Classificação: {pred}")

--- Previsões para Novas URLs ---
URL: malicious-site.phishing.ru -> Classificação: bad
URL: example.com/safe_page -> Classificação: good
URL: free-robux-generator.xyz -> Classificação: bad
URL: github.com/my-project -> Classificação: good
